Training a neuro evolution based model to learn to win 2048 game.

# Game

## Game functions

In [ ]:
import random
import numpy as np

BOARD_SIZE = 4
EMPTY_TILE = -1

In [ ]:
def spawn_tile(board):
  i = random.randrange(0, BOARD_SIZE)
  j = random.randrange(0, BOARD_SIZE)

  # Each tile independently having a value of either 2 or 4, though a 2 appears about 90% of the time and a 4 about 10% of the time
  if board[i][j] == EMPTY_TILE:
    board[i][j] = random.choices([1, 2], weights=[0.9, 0.1], k=1)[0]
    return board

  return spawn_tile(board)

In [ ]:
def generate_board():
  board = np.array([[EMPTY_TILE]*BOARD_SIZE for _ in range(BOARD_SIZE)])

  # The classic 2048 game begins on a 4×4 grid initialized with two random starting tiles placed in random empty cells.
  return spawn_tile(spawn_tile(board))

In [ ]:
def show_board(board):
  for row in board:
    for tile in row:
      if tile == EMPTY_TILE:
        print('_', end=' ')
      else:
        print(1 << tile, end=' ')
    print()
  print()

In [ ]:
board = generate_board()
show_board(board)

_ _ _ 2 
_ _ _ _ 
_ _ _ _ 
_ _ _ 2 



In [ ]:
def clean(array):
  return [tile for tile in array if tile != EMPTY_TILE]

In [ ]:
def combine(array):
  if len(array) == 1: return array

  combined_array = []
  i = 0
  while i < len(array):
    if (i+1 < len(array) and array[i] == array[i+1]):
      combined_array.append(array[i]+1)
      i += 2
    else:
      combined_array.append(array[i])
      i += 1
  return combined_array

In [ ]:
def fill(array):
  array += [EMPTY_TILE for _ in range(BOARD_SIZE-len(array))]
  return array

In [ ]:
print(combine(clean([1, 1, 1, 0])))
print(fill(combine(clean([1, 1, 1, 0]))))

[2, 1, 0]
[2, 1, 0, -1]


In [ ]:
def move_left(board):
  moved_board = []
  for row in board:
    moved_board.append(fill(combine(clean(row))))
  return np.array(moved_board)

show_board(move_left(board))

2 _ _ _ 
_ _ _ _ 
_ _ _ _ 
2 _ _ _ 



In [ ]:
def move_up(board):
  moved_board = []
  for col in range(BOARD_SIZE):
    moved_board.append(fill(combine(clean(board[:,col]))))
  return np.array(moved_board).T

show_board(move_up(board))

_ _ _ 4 
_ _ _ _ 
_ _ _ _ 
_ _ _ _ 



In [ ]:
def move_right(board):
  moved_board = []
  for row in board:
    moved_board.append(fill(combine(clean(row[::-1])))[::-1])
  return np.array(moved_board)

show_board(move_right(board))

_ _ _ 2 
_ _ _ _ 
_ _ _ _ 
_ _ _ 2 



In [ ]:
def move_down(board):
  moved_board = []
  for col in range(BOARD_SIZE):
    moved_board.append(fill(combine(clean(board[:,col][::-1])))[::-1])
  return np.array(moved_board).T

show_board(move_down(board))

_ _ _ _ 
_ _ _ _ 
_ _ _ _ 
_ _ _ 4 



In [ ]:
def move(board, dir):
  move_func = globals()['move_'+dir]
  moved_board = move_func(board)
  if np.array_equal(moved_board, board): return moved_board;

  return spawn_tile(moved_board)

for dir in ['up', 'down', 'left', 'right']:
  show_board(move(board, dir))

_ _ _ 4 
_ _ _ _ 
_ _ _ 2 
_ _ _ _ 

_ _ _ _ 
2 _ _ _ 
_ _ _ _ 
_ _ _ 4 

2 _ _ _ 
_ _ _ _ 
_ _ _ _ 
2 2 _ _ 

_ _ _ 2 
_ _ _ _ 
_ _ _ _ 
_ _ _ 2 



In [ ]:
def check_game_over(board):
  moves = ['up', 'down', 'left', 'right']
  for dir in moves:
    move_func = globals()['move_'+dir]
    if not np.array_equal(move_func(board), board):
      return False
  return True

check_game_over(np.array([[1,3,3,4],[2,3,4,5],[1,2,3,4],[2,3,4,5]]))

False

## Game loop

In [ ]:
def request_move():
  inp = int(input('Choose direction: (up->1, down->2, left->3, right->4): '))
  moves = {1: 'up', 2: 'down', 3: 'left', 4: 'right'}
  return moves[inp]

In [ ]:
def game():
  board = generate_board()

  while True:
    show_board(board)

    dir = request_move()

    board = move(board, dir)

    if check_game_over(board):
      print('Game Over!')
      break

In [ ]:
game()

4 _ _ _ 
_ _ _ 2 
_ _ _ _ 
_ _ _ _ 

Choose direction: (up->1, down->2, left->3, right->4): 


ValueError: invalid literal for int() with base 10: ''

# Model

1) Making a neuro evolution based agent
2) PPO based agent